# PageRank – vlastná implementácia

Cieľ: z vybranej stránky (**https://www.lua.org/**) vytiahnuť všetky odkazy do hĺbky 2,
vytvoriť z nich dataset hrán `zdroj -> cieľ` a ohodnotiť stránky vlastnou implementáciou
algoritmu PageRank (Google formulácia).

**Obsah**
1. Zber odkazov (crawler) → `data/raw_edges.csv`
2. Spracovanie datasetu → `data/dataset.csv`
3. Implementácia PageRank
4. Test na vzorových dátach
5. PageRank na dátach z lua.org → `data/ranking.csv`

## 1. Zber odkazov (crawler)

Postup:
- **Úroveň 0** – štartovacia stránka `https://www.lua.org/`.
- **Úroveň 1** – všetky odkazy z úvodnej stránky.
- **Úroveň 2** – všetky odkazy zo stránok úrovne 1.

Pravidlá:
- Prechádzajú sa (sťahujú) **iba stránky z domény lua.org**. Externé odkazy sa do datasetu
  zapíšu ako cieľové uzly, ale ďalej sa nesledujú.
- **Footer sa ignoruje** – pred hľadaním odkazov sa z HTML odstráni `<footer>` a všetky prvky,
  ktoré majú v `class` alebo `id` slovo `footer` (typicky obsahujú odkazy na sociálne siete).
- Ako poistka sa zahadzujú aj odkazy na **sociálne siete** (Facebook, Instagram, LinkedIn, X, …).
- Zahadzujú sa odkazy, ktoré nie sú webové stránky (`mailto:`, `javascript:`, obrázky, archívy, PDF…).
- URL sa **normalizujú**, aby rovnaká stránka nevystupovala pod viacerými názvami
  (relatívne → absolútne, bez `#fragmentu`, `http` → `https`, `.../index.html` → `.../`).
- Menu stránky (`menubar`) sa **ponecháva** – odkazy z menu sú prirodzenou súčasťou štruktúry webu.

In [ ]:
import csv
import time
from collections import deque
from pathlib import Path
from urllib.parse import urljoin, urldefrag, urlparse, urlunparse

import requests
from bs4 import BeautifulSoup

START_URL = "https://www.lua.org/"
MAX_DEPTH = 2          # počet úrovní zanorenia
DELAY = 0.3            # pauza medzi requestmi (s), aby sme server nezaťažovali
TIMEOUT = 10           # timeout jedného requestu (s)
HEADERS = {"User-Agent": "Mozilla/5.0 (PageRank school project)"}

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# domény sociálnych sietí – odkazy na ne sa zahadzujú
SOCIAL_DOMAINS = {
    "facebook.com", "instagram.com", "linkedin.com", "twitter.com", "x.com",
    "youtube.com", "tiktok.com", "pinterest.com", "reddit.com", "discord.gg",
}

# prípony súborov, ktoré nie sú HTML stránky
SKIP_EXTENSIONS = (
    ".pdf", ".jpg", ".jpeg", ".png", ".gif", ".svg", ".ico", ".webp",
    ".zip", ".gz", ".tgz", ".tar", ".bz2", ".xz", ".rar", ".7z", ".exe", ".dmg",
    ".css", ".js", ".rss", ".xml", ".txt", ".ps", ".mp3", ".mp4", ".avi",
)

START_HOST = urlparse(START_URL).netloc.lower()

In [ ]:
def bare_host(netloc: str) -> str:
    """'www.lua.org' -> 'lua.org' (na porovnávanie domén)."""
    netloc = netloc.lower().split(":")[0]
    return netloc[4:] if netloc.startswith("www.") else netloc


def is_internal(url: str) -> bool:
    return bare_host(urlparse(url).netloc) == bare_host(START_HOST)


def is_social(url: str) -> bool:
    host = bare_host(urlparse(url).netloc)
    return any(host == d or host.endswith("." + d) for d in SOCIAL_DOMAINS)


def normalize_url(href: str, base_url: str) -> str | None:
    """Prevedie href na absolútnu, normalizovanú URL. Vráti None, ak odkaz nie je stránka."""
    href = href.strip()
    if not href:
        return None
    url, _fragment = urldefrag(urljoin(base_url, href))
    parts = urlparse(url)
    if parts.scheme not in ("http", "https"):          # mailto:, javascript:, ftp:, ...
        return None
    if parts.path.lower().endswith(SKIP_EXTENSIONS):   # obrázky, archívy, pdf, ...
        return None

    scheme, netloc, path = parts.scheme, parts.netloc.lower(), parts.path or "/"
    if is_internal(url):
        # interné stránky zjednotíme na tvar https://www.lua.org/...
        scheme, netloc = "https", START_HOST
    if path.endswith("/index.html") or path.endswith("/index.htm"):
        path = path[: path.rfind("/") + 1]
    return urlunparse((scheme, netloc, path, "", parts.query, ""))


# rýchla kontrola normalizácie
for h in ["about.html", "http://lua.org/index.html", "manual/5.4/#2.1", "mailto:x@y.z",
          "ftp/lua-5.4.7.tar.gz", "https://www.puc-rio.br"]:
    print(f"{h:30} -> {normalize_url(h, START_URL)}")

In [ ]:
# štatistiky filtrovania
stats = {"footer": 0, "social": 0, "not_page": 0}


def extract_links(html: str, page_url: str) -> list[str]:
    """Vráti zoznam (aj s duplicitami) normalizovaných odkazov zo stránky, bez footeru."""
    soup = BeautifulSoup(html, "html.parser")

    # 1) odstránenie footeru
    footers = soup.find_all("footer")
    footers += soup.find_all(class_=lambda c: c and "footer" in c.lower())
    footers += soup.find_all(id=lambda i: i and "footer" in i.lower())
    for el in footers:
        if el.decomposed:            # mohol byť už odstránený ako súčasť iného footeru
            continue
        stats["footer"] += len(el.find_all("a", href=True))
        el.decompose()

    # 2) zber odkazov
    links = []
    for a in soup.find_all("a", href=True):
        url = normalize_url(a["href"], page_url)
        if url is None:
            stats["not_page"] += 1
        elif is_social(url):
            stats["social"] += 1
        else:
            links.append(url)
    return links


def fetch_html(url: str, session: requests.Session) -> str | None:
    """Stiahne stránku; vráti HTML alebo None (chyba / nie je to HTML)."""
    try:
        r = session.get(url, timeout=TIMEOUT)
    except requests.RequestException as e:
        print(f"  ! chyba: {url} ({type(e).__name__})")
        return None
    if r.status_code != 200:
        print(f"  ! HTTP {r.status_code}: {url}")
        return None
    if "text/html" not in r.headers.get("Content-Type", ""):
        return None
    r.encoding = r.encoding or r.apparent_encoding
    return r.text

In [ ]:
def crawl(start_url: str, max_depth: int) -> list[tuple[str, str]]:
    """BFS crawl. Stránky v hĺbke < max_depth sa stiahnu a ich odkazy sa uložia ako hrany."""
    start_url = normalize_url(start_url, start_url)
    edges = []
    visited = {start_url}
    queue = deque([(start_url, 0)])
    session = requests.Session()
    session.headers.update(HEADERS)

    while queue:
        url, depth = queue.popleft()
        html = fetch_html(url, session)
        time.sleep(DELAY)
        if html is None:
            continue

        links = extract_links(html, url)
        edges.extend((url, target) for target in links)
        print(f"[hĺbka {depth}] {url}  ->  {len(links)} odkazov")

        if depth + 1 < max_depth:
            for target in links:
                if is_internal(target) and target not in visited:
                    visited.add(target)
                    queue.append((target, depth + 1))
    return edges


raw_edges = crawl(START_URL, MAX_DEPTH)

In [ ]:
print(f"Surových hrán (vrátane duplicít): {len(raw_edges)}")
print(f"Stiahnutých stránok:             {len({s for s, _ in raw_edges})}")
print(f"Odfiltrované – odkazy vo footeri:  {stats['footer']}")
print(f"Odfiltrované – sociálne siete:     {stats['social']}")
print(f"Odfiltrované – nie sú stránky:     {stats['not_page']}")

with open(DATA_DIR / "raw_edges.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["source", "target"])
    writer.writerows(raw_edges)
print(f"\nUložené do {DATA_DIR / 'raw_edges.csv'}")

In [ ]:
import pandas as pd

raw_df = pd.DataFrame(raw_edges, columns=["source", "target"])
raw_df.head(20)